
Neural Transfer Através do PyTorch
=============================

Introdução
------------

Este tutorial explica como implementar o `algoritmo de estilo neural`.
Neural-Style, ou Neural-Transfer, permite tirar uma imagem e
reproduzi-lo com um novo estilo artístico. O algoritmo tira três imagens,
uma imagem de entrada, uma imagem de conteúdo e uma imagem de estilo e altera a entrada
assemelhar-se ao conteúdo da imagem-conteúdo e ao estilo artístico da imagem-estilo.


Link para o artigo original: <https://arxiv.org/pdf/1508.06576.pdf>

Princípio do algoritmo
--------------------

* Definimos duas distâncias, uma para o conteúdo
($D_C$) e um para o estilo ($D_S$). 

* $D_C$ mede a diferença do conteúdo
está entre duas imagens enquanto $D_S$ mede a diferença do estilo
entre duas imagens. 

* Então, pegamos uma terceira imagem, a entrada, e
transformamos para minimizar tanto sua distância de conteúdo com a
image de conteúdo e sua distância de estilo com a imagem de estilo. 

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from PIL import Image
import matplotlib.pyplot as plt

import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.utils import save_image

import os
import copy

import warnings
warnings.filterwarnings("ignore")

Habilitar a GPU caso disponível


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Carregando as imagens
------------------

* Agora vamos importar as imagens de estilo e conteúdo. As imagens PIL originais possuem valores entre 0 e 255, mas quando transformadas em tensores do torch, seus valores são convertidos para entre 0 e 1. 
* As imagens também precisam ser redimensionadas para terem as mesmas dimensões. 


In [3]:
def image_loader(image_name, imsize):
    loader = transforms.Compose([
      transforms.Resize(imsize),  # escalonar a imagem importada
      transforms.ToTensor()])  # transformar em tensor
    image = Image.open(image_name)
    # dimensão falsa para ajustar a dimensão requerida da rede neural
    image = loader(image).unsqueeze(0)
    return image.to(device, torch.float)

cria uma função que exibe uma imagem reconvertendo um
cópia dela para o formato PIL e exibindo a cópia usando
``plt.imshow``. 

In [4]:
unloader = transforms.ToPILImage()  # converte de volta em imagem do PIL

def imshow(tensor, title=None):
    image = tensor.cpu().clone()  # clona o tensor para não fazer mudanças nele
    image = image.squeeze(0)      # remove a dimensão falsa
    image = unloader(image)
    plt.imshow(image)
    if title is not None:
        plt.title(title)

# Loss Functions

In [5]:
class ContentLoss(nn.Module):

    def __init__(self, target,):
        super(ContentLoss, self).__init__()
        self.target = target.detach()

    def forward(self, input):
        self.loss = F.mse_loss(input, self.target)
        return input

def gram_matrix(input):
    a, b, c, d = input.size()  
    features = input.view(a * b, c * d)  
    G = torch.mm(features, features.t())  # Calcula o gram product

    return G.div(a * b * c * d) # normalização - dividir pelo total de elementos

class StyleLoss(nn.Module):

    def __init__(self, target_feature):
        super(StyleLoss, self).__init__()
        self.target = gram_matrix(target_feature).detach()

    def forward(self, input):
        G = gram_matrix(input)
        self.loss = F.mse_loss(G, self.target)
        return input           

# Importação do modelo pré-treinado

Usamos a rede VGG de 19 camadas tal como no artigo original.

A implementação do VGG no Pytorch está organizada em duas camadas sequenciais:
* ``features`` que contém as convoluções e camadas de pooling.
* ``classifier`` que contém apenas as camadas totalmente conectadas

Usaremos as features porque precisamos  da saída de cada camada de convolução para medir o content e style loss. Setamos a camada para o modo de avaliação usando ``.eval()``.




In [6]:
cnn = models.vgg19(pretrained=True).features.to(device).eval()

As redes VGG são treinadas em imagens com cada canal
normalizado pela média=[0,485, 0,456, 0,406] e desvio padrão=[0,229, 0,224, 0,225].
temos que usar esses valores para normalizar a imagem antes de enviá-la para a rede.

In [7]:
cnn_normalization_mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
cnn_normalization_std = torch.tensor([0.229, 0.224, 0.225]).to(device)

# Módulo para normalizar as imagens de entrada - faremos isso para colocar no
# nn.Sequential
class Normalization(nn.Module):
    def __init__(self, mean, std):
        super(Normalization, self).__init__()
        self.mean = torch.tensor(mean).view(-1, 1, 1)
        self.std = torch.tensor(std).view(-1, 1, 1)

    def forward(self, img):
        return (img - self.mean) / self.std

Um módulo ``Sequential`` contém uma lista ordenada de módulos filhos. Por
exemplo, ``vgg19.features`` contém uma sequência (Conv2d, ReLU, MaxPool2d,
Conv2d, ReLU…) alinhados na ordem correta de profundidade. Precisamos adicionar nossa camada de Content Loss e Style Loss imediatamente após a convolução
camada que eles estão detectando. Para fazer isso, devemos criar um novo módulo ``Sequential`` que possui módulos de Content Loss e Style Loss inseridos corretamente.

In [8]:
# Seleciona as camadas profundas para o cálculo do style/content losses:
content_layers_default = ['conv_4']
style_layers_default = ['conv_1', 'conv_2', 'conv_3', 'conv_4', 'conv_5']

def get_style_model_and_losses(cnn, normalization_mean, normalization_std,
                               style_img, content_img,
                               content_layers=content_layers_default,
                               style_layers=style_layers_default):
    # Módulo de normalização
    normalization = Normalization(normalization_mean, normalization_std).to(device)

    content_losses = []
    style_losses = []

    # Criação de um modelo nn.Sequential para a inserlção dos demais modelos
    model = nn.Sequential(normalization)

    i = 0  # incrementa sempre que acha uma convolução
    for layer in cnn.children():
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = f'conv_{i}'
        elif isinstance(layer, nn.ReLU):
            name = f'relu_{i}'
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d):
            name = f'pool_{i}'
        elif isinstance(layer, nn.BatchNorm2d):
            name = f'bn_{i}'
        else:
            raise RuntimeError(f'Unrecognized layer: {layer.__class__.__name__}')

        model.add_module(name, layer)

        if name in content_layers:
            # adiciona o content loss:
            target = model(content_img).detach()
            content_loss = ContentLoss(target)
            model.add_module("content_loss_{}".format(i), content_loss)
            content_losses.append(content_loss)

        if name in style_layers:
            # adiciona o style loss:
            target_feature = model(style_img).detach()
            style_loss = StyleLoss(target_feature)
            model.add_module("style_loss_{}".format(i), style_loss)
            style_losses.append(style_loss)

    # Apara as demais camadas após o último content e style losses
    for i in range(len(model) - 1, -1, -1):
        if isinstance(model[i], ContentLoss) or isinstance(model[i], StyleLoss):
            break

    model = model[:(i + 1)]

    return model, style_losses, content_losses

Podemos selecionar para entrada a própria imagem de conteúdo ou um ruído branco.


## Gradiente Descendente

Se usa o algortimo L-BFGS para o gradiente descendente conforme sugestão do autor do artigo.
* Vamos treinar a imagem de entrada de forma a minimizar o content e style losses.


In [9]:
def get_input_optimizer(input_img):
    # this line to show that input is a parameter that requires a gradient
    optimizer = optim.LBFGS([input_img])
    return optimizer

Por último temos uma função que realiza a transferência de aprendizado. 
* A cada iteração alimentamos a rede com uma entrada atualizada e calculamos os novos valores de loss.
* Executamos o método ``backward`` de cada módulo loss para calcular os gradientes.
* A função aninhada closure é necessária para que o otimizador reavaliar o módulo e retornar o loss.
* Em último lugar grampeamos o resultado da rede entre 0 e 1.


In [30]:
def run_style_transfer(cnn, normalization_mean, normalization_std,
                       content_img, style_img, input_img, num_steps=300,
                       style_weight=1, content_weight=1, output_dir='images/'):
  
    style_weight *= 1000000 # Esse é o valor padrão - coloquei o valor de entrada um multiplo desse valor para facilitar

    # Criar um diretório para as imagens
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Remover os arquivos da execução anterior
    for file_name in os.listdir(output_dir):
      file = output_dir + file_name
      if os.path.isfile(file):
          print(f'Apagando o arquivo {file}.')
          os.remove(file)        

    model, style_losses, content_losses = get_style_model_and_losses(cnn,
        normalization_mean, normalization_std, style_img, content_img)

    # Iremos otimizar a entrada e não o modelo
    input_img.requires_grad_(True)
    model.requires_grad_(False)

    optimizer = get_input_optimizer(input_img)

    run = [0]
    while run[0] <= num_steps:

        def closure():
            # Corrige os valores da imagem de entrada atualizada
            with torch.no_grad():
                input_img.clamp_(0, 1)

            optimizer.zero_grad()
            model(input_img)
            style_score = 0
            content_score = 0

            for sl in style_losses:
                style_score += sl.loss
            for cl in content_losses:
                content_score += cl.loss

            style_score *= style_weight
            content_score *= content_weight

            loss = style_score + content_score
            loss.backward()

            run[0] += 1
            if run[0] % 50 == 0 or run[0] == 1:
                print(100*'-')
                print(f'Epoch : {run[0]} | Style Loss : {style_score.item():4f} | Content Loss: {content_score.item():4f}')
                plt.imsave(os.path.join(output_dir,'epoch_'+str(run[0]).zfill(3)+'.jpg'), 
                               torch.clamp(input_img[0].cpu().detach().permute(1,2,0),0,1).numpy())
            return style_score + content_score

        optimizer.step(closure)

    # últoma correção - grampear entre 0 e 1
    with torch.no_grad():
        input_img.clamp_(0, 1)

    return input_img

# Execução do Algoritmo



In [33]:
#style_img = image_loader("/content/image_815.jpeg", imsize=(512,512))

#imsize = (675,1200) # paris
#imsize = (1364,2048) # marginal
#imsize = (800,1200) # puc
#imsize = (2048,1364) # paulista

imsize = (512,512)

style_img = image_loader("/content/1364px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg", imsize=imsize)
content_img = image_loader("/content/av_paulista.jpg", imsize=imsize)
input_img = content_img.clone()

'''
output = run_style_transfer(cnn, cnn_normalization_mean, cnn_normalization_std,
                            content_img, style_img, input_img, num_steps=500)
'''
# Para imagens mais abastratas aumentar o style_weigth
output = run_style_transfer(cnn, cnn_normalization_mean, cnn_normalization_std,
                            content_img, style_img, input_img, num_steps=1000,
                            style_weight=1000, content_weight=1)


# Comparação
fig,ax = plt.subplots(1,3, figsize=(20,10),facecolor='w')
ax[0].imshow(content_img[0].cpu().detach().permute(1,2,0))
ax[0].set_title("Content"), ax[0].axis('off');
ax[1].imshow(style_img[0].cpu().detach().permute(1,2,0))
ax[1].set_title("Style"), ax[1].axis('off');
ax[2].imshow(torch.clamp(output[0].cpu().detach().permute(1,2,0),0,1).numpy()), ax[2].axis('off')
ax[2].set_title("Generada");

Apagando o arquivo images/epoch_1000.jpg.
Apagando o arquivo images/epoch_450.jpg.
Apagando o arquivo images/epoch_350.jpg.
Apagando o arquivo images/epoch_050.jpg.
Apagando o arquivo images/epoch_650.jpg.
Apagando o arquivo images/epoch_100.jpg.
Apagando o arquivo images/epoch_400.jpg.
Apagando o arquivo images/epoch_600.jpg.
Apagando o arquivo images/epoch_500.jpg.
Apagando o arquivo images/epoch_150.jpg.
Apagando o arquivo images/epoch_700.jpg.
Apagando o arquivo images/epoch_900.jpg.
Apagando o arquivo images/epoch_300.jpg.
Apagando o arquivo images/epoch_950.jpg.
Apagando o arquivo images/epoch_850.jpg.
Apagando o arquivo images/epoch_750.jpg.
Apagando o arquivo images/epoch_800.jpg.
Apagando o arquivo images/epoch_550.jpg.
Apagando o arquivo images/epoch_200.jpg.
Apagando o arquivo images/epoch_250.jpg.
Apagando o arquivo images/epoch_001.jpg.
----------------------------------------------------------------------------------------------------
Epoch : 1 | Style Loss : 9067653.0000

KeyboardInterrupt: ignored

## Criar um arquivo compactado para todas as imagens geradas

In [26]:
!zip -r /content/file.zip /content/images

from google.colab import files
files.download("/content/file.zip")

updating: content/images/ (stored 0%)
updating: content/images/epoch_450.jpg (deflated 0%)
updating: content/images/epoch_350.jpg (deflated 0%)
updating: content/images/epoch_050.jpg (deflated 0%)
updating: content/images/epoch_100.jpg (deflated 0%)
updating: content/images/epoch_400.jpg (deflated 0%)
updating: content/images/epoch_500.jpg (deflated 0%)
updating: content/images/epoch_150.jpg (deflated 0%)
updating: content/images/epoch_300.jpg (deflated 0%)
updating: content/images/epoch_200.jpg (deflated 0%)
updating: content/images/epoch_250.jpg (deflated 0%)
updating: content/images/epoch_001.jpg (deflated 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>